In [38]:
import Pkg
# Pkg.update()
# Pkg.add("ProfileView")
# Pkg.add("ProfileCanvas")
using MLDatasets
using Flux: onehotbatch, onecold
using Statistics
using LinearAlgebra
using Random
using ProfileCanvas;

train_data = MLDatasets.FashionMNIST(split=:train)
test_data  = MLDatasets.FashionMNIST(split=:test)

X_train = Float32.(reshape(train_data.features, 28, 28, 1, :))
X_test  = Float32.(reshape(test_data.features, 28, 28, 1, :))

Y_train_raw = train_data.targets
Y_test_raw  = test_data.targets

Y_train = Float32.(onehotbatch(Y_train_raw, 0:9))
Y_test  = Float32.(onehotbatch(Y_test_raw, 0:9))

println("Wymiary: ", size(X_train))

Wymiary: (28, 28, 1, 60000)


In [48]:
include("clothesolver.jl") 

my_net = Chain(
  Conv((3, 3), 1 => 6, bias=false),
  MaxPool((2, 2)),
  Conv((3, 3), 6 => 16, bias=false),
  MaxPool((2, 2)),
  Flatten(),               
  Dense(784 => 84, relu),
  Dropout(0.4),
  Dense(84 => 10)
)

loss_fn = LogitCrossEntropy()

input    = GraphNode(zeros(Float32, 28, 28, 1))
target   = GraphNode(zeros(Float32, 10))
output   = my_net(input)
loss     = loss_fn(output, target)
my_model = graph(loss)


x_sample = X_train[:, :, :, 1]
y_sample = Y_train[:, 1]
forward!(my_model, input => x_sample, target => zeros(Float32, 10))

@time ProfileCanvas.@profview for _ in 1:1000
    forward!(my_model, input => x_sample, target => y_sample)
    backward!(my_model)
end


  1.900311 seconds (659.14 k allocations: 36.071 MiB, 37.43% compilation time)
